In [2]:
import polars as pl
import glob
from pathlib import Path
import polars.selectors as cs
pl.Config.set_tbl_cols(300).set_fmt_str_lengths(100).set_tbl_rows(30)

polars.config.Config

In [3]:
df_total = pl.read_parquet("../data/raw_data/*.parquet")

In [4]:
def netejar_noms(df):
    nous_noms = []
    vistos = {}

    for col in df.columns:
        nou_nom = col.split('.')[-1]
        
        #Si ja existeix, li posem un número (ex: ID_1, ID_2)
        if nou_nom in vistos:
            vistos[nou_nom] += 1
            nou_nom = f"{nou_nom}_{vistos[nou_nom]}"
        else:
            vistos[nou_nom] = 0
            
        nous_noms.append(nou_nom)
    df.columns = nous_noms
    return df

df_final = netejar_noms(df_total)

In [5]:
len(df_final)

3656443

In [6]:
selected_cols = [
    "id",
    "link_href",
    "summary",
    "title",
    "ContractFolderID",
    "ContractFolderStatusCode",
    "ContractingPartyTypeCode",
    "ContractingPartyTypeCode_listURI",
    "ID",
    "Name",
    "CityName",
    "PostalZone",
    "Line",
    "IdentificationCode",
    "Name_2",
    "Telephone",
    "ElectronicMail",
    "Name_3",
    "TypeCode",
    "TypeCode_listURI",
    "SubTypeCode",
    "SubTypeCode_listURI",
    "EstimatedOverallContractAmount",
    "TotalAmount",
    "TaxExclusiveAmount",
    "ItemClassificationCode",
    "CountrySubentityCode",
    "DurationMeasure",
    "DurationMeasure_unitCode",
    "OptionsDescription",
    "ResultCode",
    "ResultCode_listURI",
    "Name_4",
    "TotalAmount_1",
    "TaxExclusiveAmount_1",
    "ItemClassificationCode_1",
    "Description",
    "AwardDate",
    "IssueDate",
    "ID_2",
    "Name_5",
    "TaxExclusiveAmount_2",
    "PayableAmount",
    "EvaluationCriteriaTypeCode_1",
    "EvaluationCriteriaTypeCode_listURI_1",
    "EndDate",
    "EndDate_1",
    "URI_1",
    "Name_6",
    "FundingProgramCode",
    "FundingProgramCode_listURI",
    "URI_4"
]
df_c = df_final.select(selected_cols).unique()
len(df_c)

2475327

In [7]:
dic = pl.read_excel("../data/raw_data/cpv_2008_ver_2013.xlsx").with_columns(
    pl.col("CODE").str.replace_all(r"(-\d{1})", "")
)

In [8]:
df_with_codes = df_c.join(
    dic,
    left_on="ItemClassificationCode" ,
    right_on="CODE",
    how="inner"
)

In [9]:
df_with_codes.write_csv("../data/licitaciones_2012-2026.csv")


In [10]:
df_with_codes.height

2455610

In [11]:
# Què cal fer ara? Ara hem de buscar els codis CVP per determinar quines 
# licitacions son de construcció de vivenda, la que sigui, si? 

code_houseing = ["45211200", "45211000", "45211100", "45211300", "45211340"]

df_construccion = df_with_codes.filter(
    pl.col("ItemClassificationCode").str.starts_with(code_houseing[0]) |
    pl.col("ItemClassificationCode").str.starts_with(code_houseing[1]) |
    pl.col("ItemClassificationCode").str.starts_with(code_houseing[2]) |
    pl.col("ItemClassificationCode").str.starts_with(code_houseing[3]) |
    pl.col("ItemClassificationCode").str.starts_with(code_houseing[4]) 
)

df_construccion["ItemClassificationCode"].value_counts(sort=True).write_csv("../data/clean_data/construction_codes.csv")


In [12]:
len(df_construccion)

3489

In [13]:
df_construccion["ItemClassificationCode"].value_counts(sort=True).write_csv("../data/clean_data/construction_codes.csv")

In [14]:
t = df_construccion.filter(
    (pl.col("ContractFolderStatusCode") == "RES")
)
len(t)

1107

In [15]:
c = t.filter(
    (pl.col("DurationMeasure").is_not_null()) &
    (pl.col("DurationMeasure_unitCode").is_not_null())
).with_columns(
    pl.when(
        (pl.col("DurationMeasure_unitCode") == "ANN") 
    ).then(
        pl.col("DurationMeasure").cast(pl.Int32) * 365
    ).when(
        (pl.col("DurationMeasure_unitCode") == "MON")
    ).then(
        pl.col("DurationMeasure").cast(pl.Int32) * 30
    ).otherwise(
        pl.col("DurationMeasure").cast(pl.Int32)
    ), 
    pl.when(
        (pl.col("DurationMeasure_unitCode") == "ANN") | (pl.col("DurationMeasure_unitCode") == "MON")
    ).then(
        pl.lit("DAY")
    ).otherwise(
        pl.col("DurationMeasure_unitCode")
    ).alias("DurationMeasure_unitCode")
)

In [16]:
len(c)

1074

In [17]:
promotoras = c.group_by("ID_2").agg([
    pl.len().alias("count")
]).sort(by="count", descending=True)
promotoras

ID_2,count
str,u32
null,184
"""B35543958""",13
"""B10103208""",12
"""B06434641""",12
"""B05163589""",12
"""B05246400""",11
"""A73998346""",10
"""A33615931""",10
"""B05244066""",9


In [18]:
status_result = c.group_by("ResultCode").agg([
    pl.len().alias("count")
]).sort(by="count", descending=True)
status_result

ResultCode,count
str,u32
"""9""",496
"""8""",386
"""3""",174
"""4""",17
"""5""",1


In [19]:
admin_level = c.group_by("ContractingPartyTypeCode").agg([pl.len().alias("count")
]).sort(by="count", descending=True)
admin_level

ContractingPartyTypeCode,count
str,u32
"""3""",363
"""2""",358
"""5""",176
"""8""",77
"""1""",26
"""11""",24
"""7""",22
"""10""",14
"""4""",8


In [20]:
places = c.group_by("CityName").agg([pl.len().alias("count")
]).sort(by="count", descending=True)
places

CityName,count
str,u32
"""Valladolid""",175
"""Mérida""",120
"""Madrid""",100
"""Palma""",35
"""Málaga""",31
"""València""",31
"""Zaragoza""",29
"""Cáceres""",26
"""Valencia""",25


In [21]:
print(c["title"])

shape: (1_074,)
Series: 'title' [str]
[
	"vp-VA. Obras de ejecución de acondicionamiento de vivienda c/ El Plantío nº 6, 2º B, Pedrajas de San…
	"Obras de reforma y mejora de la eficiencia energética del Grupo de Viviendas VPP-PD “La Fresneda”, e…
	"Contrato de Obras de Rehabilitación Energética de 30 Viviendas en la calle Alcántara de Mérida y de …
	"construcción 10 VPO en plan parcial R2"
	"Proyecto de ejecución y estudio de seguridad y salud de obras de reparación de cubiertas en Avenida …
	"Acondicionamientos de Viviendas en Almendralejo (Badajoz) - 5 Lotes"
	"Obras para la terminación de la edificación de 39 viviendas con protección pública en alquiler, 42 g…
	"Rehabilitación das antigas vivendas dos mestres, Rúa Eduardo Blanco Amor, 10, Referencias Catastrais…
	"Construcción de un edificio piloto, de 18 viviendas de promoción pública sostenibles e innovadoras, …
	"Obras varias de reforma en el Palacio de la Capitanía General"
	"Edificio plurifamiliar para 64 VPO, locales, traster

In [22]:
mapa_numeros = {
    r"(?i)\buna?\b": "1",
    r"(?i)\bdos\b": "2",
    r"(?i)\btres\b": "3",
    r"(?i)\bcuatro\b": "4",
    r"(?i)\bcinco\b": "5",
    r"(?i)\bseis\b": "6",
    r"(?i)\bsiete\b": "7",
    r"(?i)\bocho\b": "8",
    r"(?i)\bnueve\b": "9",
    r"(?i)\bdiez\b": "10"
}

pattern = r"(?i)(?:(\d+)\s*(?:VPO|VPP|alojamientos?|viviendas?|unifamiliares?|v\.)|(?:VPO|VPP|alojamientos?|viviendas?|unifamiliares?|v\.)\s*(\d+))"
v = c.with_columns(
    pl.col("title").alias("title_clean")
)

for palabra, digito in mapa_numeros.items():
    v = v.with_columns(
        pl.col("title_clean").str.replace_all(palabra, digito)
    )

v = v.with_columns(
    pl.col("title_clean")
    .str.extract(pattern, 1)
    .fill_null(pl.col("title_clean").str.extract(pattern, 2))
    .cast(pl.Int32, strict=False)
    .alias("n_viviendas")
).drop("title_clean")


In [23]:
df_by_year = v.with_columns(
    pl.col("AwardDate").str.extract(r"\d{4}", 0).alias("year")
)

# .group_by("year").agg([
#    pl.len().alias("n_tenders")
# ]).sort(by="year", descending=True)
df_by_year

id,link_href,summary,title,ContractFolderID,ContractFolderStatusCode,ContractingPartyTypeCode,ContractingPartyTypeCode_listURI,ID,Name,CityName,PostalZone,Line,IdentificationCode,Name_2,Telephone,ElectronicMail,Name_3,TypeCode,TypeCode_listURI,SubTypeCode,SubTypeCode_listURI,EstimatedOverallContractAmount,TotalAmount,TaxExclusiveAmount,ItemClassificationCode,CountrySubentityCode,DurationMeasure,DurationMeasure_unitCode,OptionsDescription,ResultCode,ResultCode_listURI,Name_4,TotalAmount_1,TaxExclusiveAmount_1,ItemClassificationCode_1,Description,AwardDate,IssueDate,ID_2,Name_5,TaxExclusiveAmount_2,PayableAmount,EvaluationCriteriaTypeCode_1,EvaluationCriteriaTypeCode_listURI_1,EndDate,EndDate_1,URI_1,Name_6,FundingProgramCode,FundingProgramCode_listURI,URI_4,ES,n_viviendas,year
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i32,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i32,str
"""https://contrataciondelestado.es/sindicacion/licitacionesPerfilContratante/17680602""","""https://contrataciondelestado.es/wps/poc?uri=deeplink:detalle_licitacion&idEvl=wjnyTFxH69mKeVWTb9Sco…","""Id licitación: A2025/010467; Órgano de Contratación: Consejería de Medio Ambiente, Vivienda y Ordena…","""vp-VA. Obras de ejecución de acondicionamiento de vivienda c/ El Plantío nº 6, 2º B, Pedrajas de San…","""A2025/010467""","""RES""","""2""","""http://contrataciondelestado.es/codice/cl/2.10/ContractingAuthorityCode-2.10.gc""","""21038370147597""","""Consejería de Medio Ambiente, Vivienda y Ordenación del Territorio de la Junta de Castilla y León""","""Valladolid""","""47014""","""Calle Rigoberto Cortejoso, 14""","""ES""","""Consejería de Medio Ambiente, Vivienda y Ordenación del Territorio de la Junta de Castilla y León""","""983419000""","""scoad.mvo@jcyl.es""","""vp-VA. Obras de ejecución de acondicionamiento de vivienda c/ El Plantío nº 6, 2º B, Pedrajas de San…","""3""","""http://contrataciondelestado.es/codice/cl/2.08/ContractCode-2.08.gc""","""4520""","""http://contrataciondelestado.es/codice/cl/1.04/WorksContractCode-1.04.gc""","""72590""","""87833.9""","""72590""","""45211000""","""ES418""",180,"""DAY""",null,"""9""","""http://contrataciondelestado.es/codice/cl/2.09/TenderResultCode-2.09.gc""",null,null,null,null,"""Oferta con mejor relación calidad-precio.""","""2025-09-22""","""2025-10-08""","""B06909501""","""MERAKI CR S.L.""","""72590""","""87833.9""",null,null,"""2025-07-31""","""2025-07-31""","""https://contrataciondelestado.es/FileSystem/servlet/GetDocumentByIdServlet?DocumentIdParam=nrKNpaetk…","""Castilla y León""","""NO-EU""","""http://contrataciondelestado.es/codice/cl/2.08/FundingProgramCode-2.08.gc""","""https://contrataciondelestado.es/FileSystem/servlet/GetDocumentByIdServlet?DocumentIdParam=9IzzhW1Bo…","""Trabajos de construcción de inmuebles de viviendas colectivas y unifamiliares""",null,"""2025"""
"""https://contrataciondelestado.es/sindicacion/licitacionesPerfilContratante/8667338""","""https://contrataciondelestado.es/wps/poc?uri=deeplink:detalle_licitacion&idEvl=Hr%2FdpAcjAo9vYnTkQN0…","""Id licitación: A2022/000190; Órgano de Contratación: Consejería de Medio Ambiente, Vivienda y Orden…","""Obras de reforma y mejora de la eficiencia energética del Grupo de Viviendas VPP-PD “La Fresneda”, e…","""A2022/000190""","""RES""","""2""","""http://contrataciondelestado.es/codice/cl/1.04/ContractingAuthorityCode-1.04.gc""","""21038370147597""","""Consejería de Medio Ambiente, Vivienda y Ordenación del Territorio de la Junta de Castilla y León""","""Valladolid""","""47014""","""Calle Rigoberto Cortejoso, 14""","""ES""","""Consejería de Medio Ambiente, Vivienda y Ordenación del Territorio de la Junta de Castilla y León""",null,"""scoad.fyma@jcyl.es""","""Obras de reforma y mejora de la eficiencia energética del Grupo de Viviendas VPP-PD “La Fresneda”, e…","""3""","""http://contrataciondelestado.es/codice/cl/2.08/ContractCo

In [24]:
null_count = v["n_viviendas"].null_count()
print(f"Total de nulos: {null_count}")

Total de nulos: 518


In [25]:
no_estado = v.filter(
    (pl.col("ContractingPartyTypeCode") != "1")
)
len(no_estado)

1048

In [26]:
estado = v.filter(
    (pl.col("ContractingPartyTypeCode") == "1")
)

estado

id,link_href,summary,title,ContractFolderID,ContractFolderStatusCode,ContractingPartyTypeCode,ContractingPartyTypeCode_listURI,ID,Name,CityName,PostalZone,Line,IdentificationCode,Name_2,Telephone,ElectronicMail,Name_3,TypeCode,TypeCode_listURI,SubTypeCode,SubTypeCode_listURI,EstimatedOverallContractAmount,TotalAmount,TaxExclusiveAmount,ItemClassificationCode,CountrySubentityCode,DurationMeasure,DurationMeasure_unitCode,OptionsDescription,ResultCode,ResultCode_listURI,Name_4,TotalAmount_1,TaxExclusiveAmount_1,ItemClassificationCode_1,Description,AwardDate,IssueDate,ID_2,Name_5,TaxExclusiveAmount_2,PayableAmount,EvaluationCriteriaTypeCode_1,EvaluationCriteriaTypeCode_listURI_1,EndDate,EndDate_1,URI_1,Name_6,FundingProgramCode,FundingProgramCode_listURI,URI_4,ES,n_viviendas
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i32,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i32
"""https://contrataciondelestado.es/sindicacion/licitacionesPerfilContratante/17194598""","""https://contrataciondelestado.es/wps/poc?uri=deeplink:detalle_licitacion&idEvl=9vqV9cPSftCAAM7L03kM8…","""Id licitación: 2025/AR43U/00000464*; Órgano de Contratación: Intendente de San Fernando; Importe: 19…","""Obras varias de reforma en el Palacio de la Capitanía General""","""2025/AR43U/00000464*""","""RES""","""1""","""http://contrataciondelestado.es/codice/cl/2.10/ContractingAuthorityCode-2.10.gc""","""10000140000584""","""Intendente de San Fernando""","""San Fernando""","""11110""","""Arsenal de Cádiz-Base Naval de la Carraca. Sección de Contratación.""","""ES""","""Intendente de San Fernando""","""956599245""","""jucodiz@fn.mde.es""","""Obras varias de reforma en el Palacio de la Capitanía General""","""3""","""http://contrataciondelestado.es/codice/cl/2.08/ContractCode-2.08.gc""","""4545""","""http://contrataciondelestado.es/codice/cl/1.04/WorksContractCode-1.04.gc""","""196049.78""","""237220.23""","""196049.78""","""45211000""","""ES612""",120,"""DAY""",null,"""3""","""http://contrataciondelestado.es/codice/cl/2.09/TenderResultCode-2.09.gc""",null,null,null,null,null,"""2025-05-28""",null,null,null,null,null,"""5""","""http://contrataciondelestado.es/codice/cl/2.0/FinancialCapabilityTypeCode-2.0.gc""",null,"""2025-05-19""","""https://contrataciondelestado.es/FileSystem/servlet/GetDocumentByIdServlet?DocumentIdParam=PV4Eap%2B…","""Intendencia de San Fernando""","""NO-EU""","""http://contrataciondelestado.es/codice/cl/2.08/FundingProgramCode-2.08.gc""","""https://contrataciondelestado.es/FileSystem/servlet/GetDocumentByIdServlet?DocumentIdParam=v8GnsV2CM…","""Trabajos de construcción de inmuebles de viviendas colectivas y unifamiliares""",null
"""https://contrataciondelestado.es/sindicacion/licitacionesPerfilContratante/17866585""","""https://contrataciondelestado.es/wps/poc?uri=deeplink:detalle_licitacion&idEvl=g4X6YFdx9gdWhbmkna2nX…","""Id licitación: 2025/AR42U/00002109E; Órgano de Contratación: Intendente de Ferrol; Importe: 109670.4…","""EXPEDIENTE INVIED NUM. 202500000102 Obras de rehabilitación en plaza de San Vicente nº5a Pasaxe A G…","""2025/AR42U/00002109E""","""RES""","""1""","""http://contrataciondelestado.es/codice/cl/2.10/ContractingAuthorityCode-2.10.gc""","""10000140006381""","""Intendente de Ferrol""","""Ferrol (A Coruña)""","""15490""","""ARSENAL MILITAR CALLE IRMANDIÑOS S/N""","""ES""","""Intendente de Ferrol""","""981336207""","""a3jucofer@fn.mde.es""","""EXPEDIENTE INVIED NUM. 202500000102 Obras de rehabilitación en plaza de San Vicente nº5a Pasaxe A G…","""3""","""http://contrataciondelestado.es/codice/cl/2.08/ContractCode-2.08.gc""","""4510""","""http://contrataciondelestado.es/codice/cl/1.04/WorksContractCode-1.04.gc""","""109670.46""","""132701.26""","""109670.46""","""45211100""","""ES1""",60,"""DAY""",null,"""9""","""http://contrataciondelestado.es/codice/cl/2.09/TenderResultCode-2.09.gc""",null,null,null,null,"""La Mesa d

In [27]:
barcelona = c.filter(
    (pl.col("CityName") == "Barcelona")
)


In [28]:
len(barcelona)

0

In [29]:
heliopol = df_total.filter(
    (pl.col("ID_2") == "268931") 
)
heliopol

id,link_href,summary,summary_type,title,updated,ContractFolderID,ContractFolderStatusCode,ContractFolderStatusCode_languageID,ContractFolderStatusCode_listURI,ContractingPartyTypeCode,ContractingPartyTypeCode_listURI,WebsiteURI,ID,ID_schemeName,Name,CityName,PostalZone,Line,IdentificationCode,IdentificationCode_listURI,Name_1,Name_2,Telephone,Telefax,ElectronicMail,Name_3,TypeCode,TypeCode_listURI,SubTypeCode,SubTypeCode_listURI,EstimatedOverallContractAmount,EstimatedOverallContractAmount_currencyID,TotalAmount,TotalAmount_currencyID,TaxExclusiveAmount,TaxExclusiveAmount_currencyID,ItemClassificationCode,ItemClassificationCode_listURI,CountrySubentity,CountrySubentityCode,CountrySubentityCode_listURI,DurationMeasure,DurationMeasure_unitCode,OptionsDescription,ID_1,ID_schemeName_1,Name_4,TotalAmount_1,TotalAmount_currencyID_1,TaxExclusiveAmount_1,TaxExclusiveAmount_currencyID_1,ItemClassificationCode_1,ItemClassificationCode_listURI_1,ResultCode,ResultCode_listURI,Description,AwardDate,IssueDate,ID_2,ID_schemeName_2,Name_5,ProcurementProjectLotID,TaxExclusiveAmount_2,TaxExclusiveAmount_currencyID_2,PayableAmount,PayableAmount_currencyID,VariantConstraintIndicator,GuaranteeTypeCode,GuaranteeTypeCode_listURI,AmountRate,Description_1,EvaluationCriteriaTypeCode,EvaluationCriteriaTypeCode_listURI,EvaluationCriteriaTypeCode_1,EvaluationCriteriaTypeCode_listURI_1,RequirementTypeCode,RequirementTypeCode_listURI,ID_3,ProcedureCode,ProcedureCode_listURI,UrgencyCode,UrgencyCode_listURI,EndDate,EndDate_1,EndTime,ID_4,URI,DocumentHash,ID_5,URI_1,DocumentHash_1,ID_6,URI_2,DocumentHash_2,NoticeTypeCode,NoticeTypeCode_listURI,PublicationMediaName,IssueDate_1,Name_6,Name_7,Name_8,Name_9,Name_10,Name_11,IdentificationCode_1,IdentificationCode_listURI_1,Name_12,EndTime_1,Description_2,ReceivedTenderQuantity,PersonalSituation,StartDate,StartDate_1,EndDate_2,ID_7,ContractingSystemCode,ContractingSystemCode_listURI,Description_3,Rate,LowerTenderAmount,LowerTenderAmount_currencyID,HigherTenderAmount,HigherTenderAmount_currencyID,PriceRevisionFormulaDescription,Description_4,Description_5,CityName_1,ID_8,CodeValue,PostalZone_1,LiabilityAmount,LiabilityAmount_currencyID,ID_9,Note,ContractModificationDurationMeasure,ContractModificationDurationMeasure_unitCode,FinalDurationMeasure,FinalDurationMeasure_unitCode,ContractID,TaxExclusiveAmount_3,TaxExclusiveAmount_currencyID_3,TaxExclusiveAmount_4,TaxExclusiveAmount_currencyID_4,FundingProgramCode,FundingProgramCode_listURI,FundingProgram,MinimumQuantity,Description_6,LimitationDescription,ExpectedQuantity,MaximumQuantity,source_file,RequiredCurriculaIndicator,Description_7,WeightNumeric,Name_13,ActivityCode,ActivityCode_listURI,BuyerProfileURIID,ID_10,ID_schemeName_3,ID_11,ID_schemeName_4,MixContractIndicator,SMEsReceivedTenderQuantity,SMEAwardedIndicator,CountrySubentityCode_1,CountrySubentityCode_listURI_1,CountrySubentityCode_name,CityName_2,PostalZone_2,IdentificationCode_2,IdentificationCode_listURI_2,IdentificationCode_name,FundingProgramCode_name,ProcurementNationalLegislationCode,ProcurementNationalLegislationCode_listURI,ID_12,SubmissionMethodCode,SubmissionMethodCode_listURI,OverThresholdIndicator,ID_13,DocumentTypeCode,DocumentTypeCode_listURI,URI_3,FileName,Description_8,Name_14,ExecutionRequirementCode,ExecutionRequirementCode_listURI,Description_9,AwardingCriteriaTypeCode,AwardingCriteriaTypeCode_listURI,AwardingCriteriaSubTypeCode,AwardingCriteriaSubTypeCode_listURI,EndpointID,AuctionConstraintIndicator,ReceivedAppealQuantity,DocumentTypeCode_1,DocumentTypeCode_listURI_1,DocumentTypeCode_name,URI_4,FileName_1,UUID,UUID_schemeName,AwardedOwnerNationalityCode,AwardedOwnerNationalityCode_listURI,Note_1,AgencyID,SendDate,SendTime,EvaluationCriteriaTypeCode_2,EvaluationCriteriaTypeCode_listURI_2,Description_10,EvaluationCriteriaTypeCode_3,EvaluationCriteriaTypeCode_listURI_3,Description_11,CountrySubentity_1,CountrySubentityCode_2,CountrySubentityCode_listURI_2,IdentificationCode_3,Identificatio

In [30]:
v.write_csv("../data/clean_data/licitaciones_construccion_vivienda.csv")

In [38]:
# Creamos y exportamos el dataset final con todos los campos que necesitemos.
df_f = df_by_year.unique(subset=["id"]).with_columns(
    pl.col("CityName").replace(
            {"València":"Valencia"},
            {"Santa Cruz de Tenerife":"Tenerife"})
        ).select([
        pl.col("id").alias("id"),
        pl.col("title"),
        pl.col("link_href"),
        pl.col("ResultCode").alias("status"),
        pl.col("year"),
        pl.col("EstimatedOverallContractAmount").alias("estimatedAmount"),
        pl.col("PayableAmount").alias("payableAmount"),
        pl.col("ContractingPartyTypeCode").alias("organo"),
        pl.col("PostalZone").alias("PostalCode"),
        pl.col("CityName").alias("city"),
        pl.col("DurationMeasure").alias("duration"),
        pl.col("ID_2").alias("cif_provider"),
        pl.col("Name_5").alias("name_provider"),
        pl.col("n_viviendas")
    ]).sort(by="status", descending=True)

df_f.write_json("../my-app/src/lib/components/datos/test.json")
df_f.write_csv("../my-app/src/lib/components/datos/test.csv")

In [ ]:
len(df_f)

In [ ]:
count = df_f["cif_provider"].len()
count

In [ ]:
unique_count = df_f["cif_provider"].n_unique()
unique_count

In [ ]:
payableAmount_by_year = df_f.group_by("year").agg([
    pl.col("payableAmount").str.to_decimal(scale=1).sum()
])
payableAmount_by_year